# 🐭 Análisis de Videos de Chinchillas
### Extracción de fecha/hora sobreimpresa en videos por guarida

**Estructura esperada en Drive:**
```
Mi unidad/
  └── Chinchillas/          ← ajusta RUTA_BASE abajo
        ├── G1/
        │     ├── G1/       (subcarpetas opcionales)
        │     ├── G1+1/
        │     └── G1+2/
        ├── G2/
        ├── G3/
        ├── G4/
        ├── G5/
        ├── G6/
        └── G7/
```

## Celda 1 — Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive montado')

## Celda 2 — Instalar dependencias (ejecutar solo una vez)

In [ ]:
!apt-get install -q tesseract-ocr tesseract-ocr-spa 2>/dev/null
!pip install -q pytesseract opencv-python-headless
print('✅ Dependencias instaladas')

## Celda 3 — ⚙️ CONFIGURACIÓN (ajusta aquí)

In [ ]:
import os

# ─── AJUSTA ESTA RUTA ──────────────────────────────────────────────────────────
# Ruta a la carpeta que contiene G1, G2, G3 ... G7
RUTA_BASE = '/content/drive/MyDrive/Chinchillas'
# ──────────────────────────────────────────────────────────────────────────────

# Dónde se guarda el checkpoint (para no perder progreso si Colab se corta)
CHECKPOINT_CSV = os.path.join(RUTA_BASE, '_checkpoint_registros.csv')

# Extensiones de video reconocidas
EXTENSIONES_VIDEO = ('.mp4', '.avi', '.mov', '.mkv', '.MP4', '.AVI', '.MOV', '.MKV')

# Número de frames a revisar por video en busca de la fecha
FRAMES_A_REVISAR = 8

# ¿Dónde está el timestamp en el frame? Opciones: 'auto', 'top', 'bottom', 'top_half', 'bottom_half'
# 'auto'  → revisa el frame completo (más lento pero seguro)
# 'top'   → solo el 15% superior del frame
# 'bottom'→ solo el 15% inferior del frame
ZONA_TIMESTAMP = 'auto'

print(f'📁 Ruta base: {RUTA_BASE}')
if os.path.isdir(RUTA_BASE):
    contenido = sorted(os.listdir(RUTA_BASE))
    print(f'✅ Carpeta encontrada. Contiene: {contenido}')
else:
    print('❌ ERROR: La carpeta no existe. Verifica RUTA_BASE.')

## Celda 4 — Previsualizar primer frame (diagnóstico)
Ejecuta esto para ver dónde está la fecha en tus videos.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import glob

# Encuentra el primer video disponible automáticamente
patron = os.path.join(RUTA_BASE, '**', '*')
todos = glob.glob(patron, recursive=True)
videos = [f for f in todos if f.endswith(EXTENSIONES_VIDEO)]

if not videos:
    print('⚠️  No se encontraron videos. Verifica RUTA_BASE.')
else:
    video_prueba = videos[0]
    print(f'Usando: {video_prueba}')

    cap = cv2.VideoCapture(video_prueba)
    ret, frame = cap.read()
    cap.release()

    if ret:
        alto, ancho = frame.shape[:2]
        print(f'Resolución: {ancho}x{alto} px')

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Frame completo
        axes[0].imshow(frame_rgb)
        axes[0].set_title('Frame completo')
        axes[0].axis('off')

        # Zona superior (15%)
        zona_sup = frame_rgb[:int(alto*0.15), :]
        axes[1].imshow(zona_sup)
        axes[1].set_title(f'Zona SUPERIOR (top 15%)')
        axes[1].axis('off')

        # Zona inferior (15%)
        zona_inf = frame_rgb[int(alto*0.85):, :]
        axes[2].imshow(zona_inf)
        axes[2].set_title(f'Zona INFERIOR (bottom 15%)')
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()
        print('👆 Observa en qué zona aparece la fecha y actualiza ZONA_TIMESTAMP en Celda 3')
    else:
        print('❌ No se pudo leer el frame del video')

## Celda 5 — Funciones de extracción

In [ ]:
import cv2
import pytesseract
import re
import numpy as np

# Patrones de fecha/hora típicos de cámaras trampa
PATRONES_FECHA_HORA = [
    # YYYY-MM-DD HH:MM:SS  o  YYYY/MM/DD HH:MM:SS
    (r'(\d{4}[-/]\d{2}[-/]\d{2})\s+(\d{2}:\d{2}(?::\d{2})?)', '%Y-%m-%d', '%H:%M:%S'),
    # DD-MM-YYYY HH:MM:SS  o  DD/MM/YYYY HH:MM:SS
    (r'(\d{2}[-/]\d{2}[-/]\d{4})\s+(\d{2}:\d{2}(?::\d{2})?)', '%d-%m-%Y', '%H:%M:%S'),
    # DD-MM-YY HH:MM:SS
    (r'(\d{2}[-/]\d{2}[-/]\d{2})\s+(\d{2}:\d{2}(?::\d{2})?)', '%d-%m-%y', '%H:%M:%S'),
    # Solo fecha YYYY-MM-DD
    (r'(\d{4}[-/]\d{2}[-/]\d{2})', '%Y-%m-%d', None),
]

def recortar_zona(frame, zona):
    """Recorta el frame según la zona de interés."""
    alto, ancho = frame.shape[:2]
    if zona == 'top':
        return frame[:int(alto * 0.15), :]
    elif zona == 'bottom':
        return frame[int(alto * 0.85):, :]
    elif zona == 'top_half':
        return frame[:alto // 2, :]
    elif zona == 'bottom_half':
        return frame[alto // 2:, :]
    else:  # 'auto' o cualquier otro
        return frame

def mejorar_para_ocr(img_gris):
    """Aplica preprocesamiento para mejorar el OCR."""
    # Escalar al doble para mejor OCR
    img = cv2.resize(img_gris, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    # Umbralización adaptativa
    img = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY, 11, 2)
    return img

def extraer_fecha_hora_ocr(texto):
    """Busca patrones de fecha y hora en el texto OCR."""
    for patron, *_ in PATRONES_FECHA_HORA:
        match = re.search(patron, texto)
        if match:
            fecha = match.group(1).replace('/', '-')
            hora = match.group(2) if match.lastindex >= 2 else None
            return fecha, hora
    return None, None

def extraer_fecha_hora_video(ruta_video, zona=ZONA_TIMESTAMP, num_frames=FRAMES_A_REVISAR):
    """
    Abre el video y extrae la fecha/hora del primer frame donde se encuentre.
    Devuelve (fecha_str, hora_str, duracion_seg).
    """
    cap = cv2.VideoCapture(ruta_video)
    if not cap.isOpened():
        return None, None, 0

    fps = cap.get(cv2.CAP_PROP_FPS) or 1
    total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    duracion_seg = round(total_frames / fps) if fps > 0 else 0

    fecha, hora = None, None
    config_tess = '--psm 6 --oem 3'

    for i in range(num_frames):
        ret, frame = cap.read()
        if not ret:
            break

        zona_img = recortar_zona(frame, zona)
        gris = cv2.cvtColor(zona_img, cv2.COLOR_BGR2GRAY)

        # Intento 1: imagen original en gris
        texto = pytesseract.image_to_string(gris, config=config_tess)
        fecha, hora = extraer_fecha_hora_ocr(texto)
        if fecha:
            break

        # Intento 2: con preprocesamiento
        gris_mejorado = mejorar_para_ocr(gris)
        texto2 = pytesseract.image_to_string(gris_mejorado, config=config_tess)
        fecha, hora = extraer_fecha_hora_ocr(texto2)
        if fecha:
            break

        # Intento 3: imagen invertida (texto blanco sobre fondo oscuro)
        gris_inv = cv2.bitwise_not(gris)
        texto3 = pytesseract.image_to_string(gris_inv, config=config_tess)
        fecha, hora = extraer_fecha_hora_ocr(texto3)
        if fecha:
            break

    cap.release()
    return fecha, hora, duracion_seg

def listar_videos_en_carpeta(ruta_carpeta):
    """Devuelve lista de rutas de todos los videos en la carpeta y subcarpetas."""
    videos = []
    for raiz, dirs, archivos in os.walk(ruta_carpeta):
        dirs.sort()  # orden consistente
        for archivo in sorted(archivos):
            if archivo.endswith(EXTENSIONES_VIDEO):
                videos.append(os.path.join(raiz, archivo))
    return videos

print('✅ Funciones cargadas')

## Celda 6 — Contar videos antes de procesar

In [ ]:
import os

print('📊 Conteo de videos por guarida:\n')
total_global = 0

guaridas_principales = sorted([
    d for d in os.listdir(RUTA_BASE)
    if os.path.isdir(os.path.join(RUTA_BASE, d)) and not d.startswith('_')
])

for guarida in guaridas_principales:
    ruta_guarida = os.path.join(RUTA_BASE, guarida)
    videos = listar_videos_en_carpeta(ruta_guarida)
    print(f'  {guarida}: {len(videos)} videos')
    total_global += len(videos)

print(f'\n  TOTAL: {total_global} videos')
print(f'\n⏱️  Tiempo estimado (~3 seg/video): {round(total_global * 3 / 60, 1)} minutos')

## Celda 7 — 🚀 Procesar todos los videos (con checkpoint)
Si Colab se corta, vuelve a ejecutar esta celda: retomará donde quedó.

In [ ]:
import pandas as pd
import time

# Cargar checkpoint si existe (para retomar en caso de interrupción)
if os.path.exists(CHECKPOINT_CSV):
    df_prev = pd.read_csv(CHECKPOINT_CSV)
    ya_procesados = set(df_prev['Ruta_completa'].tolist())
    registros = df_prev.to_dict('records')
    print(f'♻️  Retomando desde checkpoint: {len(registros)} videos ya procesados')
else:
    ya_procesados = set()
    registros = []
    print('🆕 Iniciando desde cero')

guaridas_principales = sorted([
    d for d in os.listdir(RUTA_BASE)
    if os.path.isdir(os.path.join(RUTA_BASE, d)) and not d.startswith('_')
])

GUARDAR_CADA_N = 10  # guardar checkpoint cada N videos
contador = 0

for guarida in guaridas_principales:
    ruta_guarida = os.path.join(RUTA_BASE, guarida)
    videos = listar_videos_en_carpeta(ruta_guarida)

    for ruta_video in videos:
        if ruta_video in ya_procesados:
            continue  # ya procesado, saltar

        # Determinar subcarpeta relativa
        rel = os.path.relpath(ruta_video, ruta_guarida)
        partes = rel.split(os.sep)
        subcarpeta = partes[-2] if len(partes) > 1 else ''
        archivo = partes[-1]

        etiqueta = f'{guarida}/{subcarpeta}/{archivo}' if subcarpeta else f'{guarida}/{archivo}'
        print(f'  🎬 {etiqueta}... ', end='', flush=True)

        t0 = time.time()
        try:
            fecha, hora, duracion = extraer_fecha_hora_video(ruta_video)
            estado = '✓' if fecha else '?'
        except Exception as e:
            fecha, hora, duracion = None, None, 0
            estado = f'ERROR: {e}'

        elapsed = round(time.time() - t0, 1)
        print(f'{estado} {fecha} {hora} ({duracion}s) [{elapsed}s]')

        registros.append({
            'Guarida': guarida,
            'Subcarpeta': subcarpeta,
            'Archivo': archivo,
            'Fecha': fecha,
            'Hora': hora,
            'Duracion_seg': duracion,
            'Ruta_completa': ruta_video
        })
        ya_procesados.add(ruta_video)
        contador += 1

        # Guardar checkpoint periódicamente
        if contador % GUARDAR_CADA_N == 0:
            pd.DataFrame(registros).to_csv(CHECKPOINT_CSV, index=False)
            print(f'  💾 Checkpoint guardado ({len(registros)} videos procesados)')

# Guardar checkpoint final
pd.DataFrame(registros).to_csv(CHECKPOINT_CSV, index=False)
print(f'\n✅ Procesamiento completo: {len(registros)} videos en total')

## Celda 8 — Generar tablas y exportar CSV

In [ ]:
import pandas as pd

df = pd.DataFrame(registros)

print(f'Total de videos: {len(df)}')
print(f'Con fecha detectada: {df["Fecha"].notna().sum()}')
print(f'Sin fecha (revisar): {df["Fecha"].isna().sum()}')
print()

# ── Tabla 1: registro completo ─────────────────────────────────────────────────
ruta_completo = os.path.join(RUTA_BASE, 'registro_completo.csv')
df.to_csv(ruta_completo, index=False, encoding='utf-8-sig')
print(f'💾 Guardado: registro_completo.csv')

# ── Tabla 2: visitas por guarida por día ───────────────────────────────────────
df_con_fecha = df[df['Fecha'].notna()].copy()

if not df_con_fecha.empty:
    tabla_diaria = df_con_fecha.groupby(['Guarida', 'Subcarpeta', 'Fecha']).agg(
        Num_videos=('Archivo', 'count'),
        Duracion_total_min=('Duracion_seg', lambda x: round(x.sum() / 60, 1)),
        Primer_registro=('Hora', 'min'),
        Ultimo_registro=('Hora', 'max')
    ).reset_index()
    tabla_diaria = tabla_diaria.sort_values(['Guarida', 'Subcarpeta', 'Fecha'])

    ruta_diaria = os.path.join(RUTA_BASE, 'datos_diarios.csv')
    tabla_diaria.to_csv(ruta_diaria, index=False, encoding='utf-8-sig')
    print(f'💾 Guardado: datos_diarios.csv')
    print()
    print('── Resumen por guarida ──')
    print(tabla_diaria.to_string(index=False))
else:
    print('⚠️  No se pudo extraer fecha de ningún video.')
    print('   → Ejecuta la Celda 4 (diagnóstico) y ajusta ZONA_TIMESTAMP')
    print('   → Revisa que el video tenga fecha sobreimpresa visible')

# ── Tabla 3: videos sin fecha (para revisión manual) ──────────────────────────
df_sin_fecha = df[df['Fecha'].isna()]
if not df_sin_fecha.empty:
    ruta_problemas = os.path.join(RUTA_BASE, 'videos_sin_fecha.csv')
    df_sin_fecha.to_csv(ruta_problemas, index=False, encoding='utf-8-sig')
    print(f'\n⚠️  {len(df_sin_fecha)} videos sin fecha guardados en: videos_sin_fecha.csv')

## Celda 9 — Diagnóstico: inspeccionar un video sin fecha
Si hay videos sin fecha, esto te ayuda a ver qué ve el OCR.

In [ ]:
import cv2, pytesseract, matplotlib.pyplot as plt

# Cambia esta ruta por la de un video sin fecha
VIDEO_DIAGNOSTICO = registros[0]['Ruta_completa'] if registros else ''

if VIDEO_DIAGNOSTICO and os.path.exists(VIDEO_DIAGNOSTICO):
    cap = cv2.VideoCapture(VIDEO_DIAGNOSTICO)
    ret, frame = cap.read()
    cap.release()

    if ret:
        alto, ancho = frame.shape[:2]
        gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        zonas = {
            'Completo': gris,
            'Superior 15%': gris[:int(alto*0.15), :],
            'Inferior 15%': gris[int(alto*0.85):, :],
            'Inferior 15% (invertido)': cv2.bitwise_not(gris[int(alto*0.85):, :]),
        }

        config = '--psm 6 --oem 3'
        for nombre, img in zonas.items():
            texto = pytesseract.image_to_string(img, config=config).strip()
            print(f'--- {nombre} ---')
            print(repr(texto[:200]))
            print()

        fig, axes = plt.subplots(1, 4, figsize=(20, 4))
        for ax, (nombre, img) in zip(axes, zonas.items()):
            ax.imshow(img, cmap='gray')
            ax.set_title(nombre, fontsize=9)
            ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print('No hay videos para diagnosticar')